##Transform Drivers Data

1. Read bronze drivers table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (driverId->driver_id, dateOfBirth->date_of_birth)
4. Concatenate name.givenName and name.familyName to create a new column driver_name and transform the values to Title Case
5. Remove duplicate records
6. Transform values of columns nationality to Title Case
7. Write the transformed data to silver drivers table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

####Step 1: Read the bronze drivers table

In [0]:
drivers_df = (
    spark.table(bronze_table)
        .filter((F.col("batch_id") == v_batch_id))
)

In [0]:
display(drivers_df)

driverId,name,dateOfBirth,nationality,url,ingestion_timestamp,source_file,batch_id
abate,"List(carlo, abate)",1932-07-10,italian,http://en.wikipedia.org/wiki/Carlo_Mario_Abate,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
abecassis,"List(george, abecassis)",1913-03-21,british,http://en.wikipedia.org/wiki/George_Abecassis,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
acheson,"List(kenny, acheson)",1957-11-27,british,http://en.wikipedia.org/wiki/Kenny_Acheson,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
adams,"List(philippe, adams)",1969-11-19,belgian,http://en.wikipedia.org/wiki/Philippe_Adams,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
ader,"List(walt, ader)",1913-12-15,american,http://en.wikipedia.org/wiki/Walt_Ader,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
adolff,"List(kurt, adolff)",1921-11-05,german,http://en.wikipedia.org/wiki/Kurt_Adolff,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
agabashian,"List(fred, agabashian)",1913-08-21,american,http://en.wikipedia.org/wiki/Fred_Agabashian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
ahrens,"List(kurt, ahrens)",1940-04-19,german,"http://en.wikipedia.org/wiki/Kurt_Ahrens,_Jr.",2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
aitken,"List(jack, aitken)",1995-09-23,british,http://en.wikipedia.org/wiki/Jack_Aitken,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
albers,"List(christijan, albers)",1979-04-16,dutch,http://en.wikipedia.org/wiki/Christijan_Albers,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01


####Step 2: Keep only the coluumns required for analysis (Drop url column)

There are 2 ways to do this... one by selecting the required columns and the other by dropping the unwanted columns

In [0]:
##This method uses dropping of the non-required columns
from pyspark.sql import functions as F

drivers_dropped_df = drivers_df.drop("url")

display(drivers_dropped_df)

driverId,name,dateOfBirth,nationality,ingestion_timestamp,source_file,batch_id
abate,"List(carlo, abate)",1932-07-10,italian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
abecassis,"List(george, abecassis)",1913-03-21,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
acheson,"List(kenny, acheson)",1957-11-27,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
adams,"List(philippe, adams)",1969-11-19,belgian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
ader,"List(walt, ader)",1913-12-15,american,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
adolff,"List(kurt, adolff)",1921-11-05,german,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
agabashian,"List(fred, agabashian)",1913-08-21,american,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
ahrens,"List(kurt, ahrens)",1940-04-19,german,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
aitken,"List(jack, aitken)",1995-09-23,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
albers,"List(christijan, albers)",1979-04-16,dutch,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01


####Step 3: Standardizing Column names using snake case

In [0]:
drivers_renamed_df = (
    drivers_dropped_df
        .withColumnsRenamed({
            "driverId": "driver_id",
            "dateOfBirth": "date_of_birth"})
)

In [0]:
display(drivers_renamed_df)

driver_id,name,date_of_birth,nationality,ingestion_timestamp,source_file,batch_id
abate,"List(carlo, abate)",1932-07-10,italian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
abecassis,"List(george, abecassis)",1913-03-21,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
acheson,"List(kenny, acheson)",1957-11-27,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
adams,"List(philippe, adams)",1969-11-19,belgian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
ader,"List(walt, ader)",1913-12-15,american,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
adolff,"List(kurt, adolff)",1921-11-05,german,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
agabashian,"List(fred, agabashian)",1913-08-21,american,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
ahrens,"List(kurt, ahrens)",1940-04-19,german,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
aitken,"List(jack, aitken)",1995-09-23,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
albers,"List(christijan, albers)",1979-04-16,dutch,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01


####Step 4: Concatenate givenName and familyName to make new column driver_name and make the values Title Case

In [0]:
drivers_concatenated_df = (
    drivers_renamed_df
        .withColumn("driver_name",
                    F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))))
        .drop("name")
)
display(drivers_concatenated_df)

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,batch_id,driver_name
abate,1932-07-10,italian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Carlo Abate
abecassis,1913-03-21,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,George Abecassis
acheson,1957-11-27,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kenny Acheson
adams,1969-11-19,belgian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Philippe Adams
ader,1913-12-15,american,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Walt Ader
adolff,1921-11-05,german,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kurt Adolff
agabashian,1913-08-21,american,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Fred Agabashian
ahrens,1940-04-19,german,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kurt Ahrens
aitken,1995-09-23,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Jack Aitken
albers,1979-04-16,dutch,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Christijan Albers


####Step 5: Remove duplicate records

In [0]:
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])
display(drivers_distinct_df)

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,batch_id,driver_name
abate,1932-07-10,italian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Carlo Abate
abecassis,1913-03-21,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,George Abecassis
acheson,1957-11-27,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kenny Acheson
adams,1969-11-19,belgian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Philippe Adams
ader,1913-12-15,american,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Walt Ader
adolff,1921-11-05,german,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kurt Adolff
agabashian,1913-08-21,american,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Fred Agabashian
ahrens,1940-04-19,german,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kurt Ahrens
aitken,1995-09-23,british,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Jack Aitken
albers,1979-04-16,dutch,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Christijan Albers


####Step 6: Transform the values of nationality to Title Case

In [0]:
drivers_final_df = (
    drivers_distinct_df
        .withColumn("nationality", F.initcap(F.col("nationality")))
)

display(drivers_final_df)

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,batch_id,driver_name
abate,1932-07-10,Italian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Carlo Abate
abecassis,1913-03-21,British,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,George Abecassis
acheson,1957-11-27,British,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kenny Acheson
adams,1969-11-19,Belgian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Philippe Adams
ader,1913-12-15,American,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Walt Ader
adolff,1921-11-05,German,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kurt Adolff
agabashian,1913-08-21,American,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Fred Agabashian
ahrens,1940-04-19,German,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kurt Ahrens
aitken,1995-09-23,British,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Jack Aitken
albers,1979-04-16,Dutch,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Christijan Albers


####Step 7: Write the transformed data to the silver 'constructors' table

In [0]:
write_to_silver(
    input_df = drivers_final_df,
    target_table = silver_table,
    merge_condition = "t.driver_id = s.driver_id",
    columns_to_update = [
        "driver_id",
        "date_of_birth",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id",
        "driver_name"
    ]
)

In [0]:
display(spark.table(silver_table))

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,batch_id,driver_name,created_timestamp,updated_timestamp
ahrens,1940-04-19,German,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kurt Ahrens,2026-08-05T15:10:35.473Z,2026-08-05T15:11:00.689Z
barilla,1961-04-20,Italian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Paolo Barilla,2026-08-05T15:10:35.473Z,2026-08-05T15:11:00.689Z
bayol,1914-02-28,French,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Élie Bayol,2026-08-05T15:10:35.473Z,2026-08-05T15:11:00.689Z
birger,1924-01-07,Argentine,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Pablo Birger,2026-08-05T15:10:35.473Z,2026-08-05T15:11:00.689Z
borgudd,1946-11-25,Swedish,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Slim Borgudd,2026-08-05T15:10:35.473Z,2026-08-05T15:11:00.689Z
branca,1916-09-15,Swiss,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Toni Branca,2026-08-05T15:10:35.473Z,2026-08-05T15:11:00.689Z
brown,1949-12-24,Australian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Warwick Brown,2026-08-05T15:10:35.473Z,2026-08-05T15:11:00.689Z
christie,1924-04-04,American,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Bob Christie,2026-08-05T15:10:35.473Z,2026-08-05T15:11:00.689Z
clark,1936-03-04,British,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Jim Clark,2026-08-05T15:10:35.473Z,2026-08-05T15:11:00.689Z
george_connor,1906-08-16,American,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,George Connor,2026-08-05T15:10:35.473Z,2026-08-05T15:11:00.689Z
